## Célula 1 - Imports

Esta célula carrega apenas bibliotecas e utilitários. A lógica começa nas células seguintes.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import random
import shutil
import zipfile
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping

import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

try:
    import torch
except Exception:
    torch = None

try:
    from IPython.display import display
except Exception:
    display = print


## Célula 2 - Base do Dataset

Aqui ficam a configuração global, a validação do `.zip` e a análise da distribuição das classes.

In [2]:
PROJECT_ROOT = Path.cwd().resolve()
TRAIN_ZIP_PATH = PROJECT_ROOT / 'train.zip'
VAL_ZIP_PATH = PROJECT_ROOT / 'val.zip'
TEST_ZIP_PATH = PROJECT_ROOT / 'test.zip'
DATASETS_DIR = PROJECT_ROOT / 'datasets'
TRAIN_DIR = DATASETS_DIR / 'train'
VAL_DIR = DATASETS_DIR / 'val'
TEST_DIR = DATASETS_DIR / 'test'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
LOGGER = logging.getLogger('notebook')


def ensure_zip_exists(zip_path: Path) -> None:
    if not zip_path.exists():
        raise FileNotFoundError(f'Arquivo zip nao encontrado: {zip_path.resolve()}')


def extract_dataset(zip_path: Path, target_dir: Path) -> Path:
    '''Extract one split from its own zip and return the folder that contains images/labels.'''
    ensure_zip_exists(zip_path)
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(target_dir)

    candidates = [target_dir] + [p for p in target_dir.rglob('*') if p.is_dir()]
    dataset_roots = [p for p in candidates if (p / 'images').is_dir() and (p / 'labels').is_dir()]
    if not dataset_roots:
        raise ValueError(f'{zip_path.name} nao contem uma estrutura valida com images/ e labels/.')
    dataset_root = min(dataset_roots, key=lambda p: len(p.relative_to(target_dir).parts))
    if not (dataset_root / 'classes.txt').exists():
        label_classes = dataset_root / 'labels' / 'classes.txt'
        image_classes = dataset_root / 'images' / 'classes.txt'
        if label_classes.exists():
            shutil.copy2(label_classes, dataset_root / 'classes.txt')
        elif image_classes.exists():
            shutil.copy2(image_classes, dataset_root / 'classes.txt')
        else:
            raise ValueError(f'{zip_path.name} nao contem classes.txt na raiz do split.')
    return dataset_root


def load_class_names(classes_file: Path) -> list[str]:
    if not classes_file.exists():
        raise FileNotFoundError(f'classes.txt nao encontrado: {classes_file.resolve()}')
    names = [line.strip() for line in classes_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    if not names:
        raise ValueError(f'classes.txt vazio: {classes_file.resolve()}')
    return names


def compare_class_lists(*class_lists: list[str]) -> None:
    reference = class_lists[0]
    for index, names in enumerate(class_lists[1:], start=2):
        if names != reference:
            raise ValueError(f'Classes inconsistentes entre splits. Split 1={reference}; split {index}={names}')


def read_label_rows(label_file: Path) -> list[tuple[int, float, float, float, float]]:
    rows: list[tuple[int, float, float, float, float]] = []
    if not label_file.exists() or label_file.name == 'classes.txt':
        return rows
    for line_number, raw_line in enumerate(label_file.read_text(encoding='utf-8').splitlines(), start=1):
        raw_line = raw_line.strip()
        if not raw_line:
            continue
        parts = raw_line.split()
        if len(parts) < 5:
            raise ValueError(f'Label invalido em {label_file.resolve()} linha {line_number}: {raw_line}')
        class_id = int(float(parts[0]))
        coords = tuple(round(float(value), 6) for value in parts[1:5])
        rows.append((class_id, *coords))
    return rows


def read_label_classes(label_file: Path) -> list[int]:
    return [row[0] for row in read_label_rows(label_file)]


def collect_image_hashes(split_root: Path) -> dict[str, list[Path]]:
    hashes: dict[str, list[Path]] = {}
    for image_path in sorted((split_root / 'images').rglob('*')):
        if image_path.suffix.lower() in IMAGE_EXTENSIONS:
            digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
            hashes.setdefault(digest, []).append(image_path)
    return hashes


def collect_coordinate_signatures(split_root: Path) -> dict[tuple[tuple[int, float, float, float, float], ...], list[Path]]:
    signatures: dict[tuple[tuple[int, float, float, float, float], ...], list[Path]] = {}
    for label_path in sorted((split_root / 'labels').rglob('*.txt')):
        if label_path.name == 'classes.txt':
            continue
        rows = tuple(sorted(read_label_rows(label_path)))
        if rows:
            signatures.setdefault(rows, []).append(label_path)
    return signatures


def overlap_rows(kind: str, split_a: str, split_b: str, left: dict[Any, list[Path]], right: dict[Any, list[Path]]) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    for signature in sorted(set(left).intersection(right), key=str):
        for left_path in left[signature]:
            for right_path in right[signature]:
                rows.append({
                    'tipo': kind,
                    'split_a': split_a,
                    'arquivo_a': str(left_path.resolve()),
                    'split_b': split_b,
                    'arquivo_b': str(right_path.resolve()),
                })
    return rows


def detect_data_leakage(train_root: Path, val_root: Path, test_root: Path) -> None:
    '''Detect leakage. Train/val image overlap warns unless coordinates also overlap; any test overlap fails.'''
    roots = {'train': train_root, 'val': val_root, 'test': test_root}
    image_hashes = {name: collect_image_hashes(root) for name, root in roots.items()}
    coordinate_signatures = {name: collect_coordinate_signatures(root) for name, root in roots.items()}

    train_val_images = overlap_rows('imagem_duplicada', 'train', 'val', image_hashes['train'], image_hashes['val'])
    train_val_coords = overlap_rows('coordenadas_duplicadas', 'train', 'val', coordinate_signatures['train'], coordinate_signatures['val'])
    test_images = overlap_rows('imagem_duplicada', 'train', 'test', image_hashes['train'], image_hashes['test'])
    test_images += overlap_rows('imagem_duplicada', 'val', 'test', image_hashes['val'], image_hashes['test'])
    test_coords = overlap_rows('coordenadas_duplicadas', 'train', 'test', coordinate_signatures['train'], coordinate_signatures['test'])
    test_coords += overlap_rows('coordenadas_duplicadas', 'val', 'test', coordinate_signatures['val'], coordinate_signatures['test'])

    if test_images or test_coords:
        report = pd.DataFrame(test_images + test_coords)
        print('Data leakage envolvendo TEST detectado:')
        display(report)
        raise ValueError('Data leakage envolvendo TEST. TEST nao pode aparecer no treino, validacao, Optuna ou selecao.')

    if train_val_coords:
        report = pd.DataFrame(train_val_coords)
        print('Data leakage entre TRAIN e VAL com as mesmas coordenadas detectado:')
        display(report)
        raise ValueError('Data leakage entre TRAIN e VAL: as mesmas coordenadas/anotacoes aparecem nos dois splits.')

    if train_val_images:
        print('Aviso: possivel data leakage entre TRAIN e VAL por imagem repetida, mas sem conflito de coordenadas iguais.')
        display(pd.DataFrame(train_val_images))
    else:
        print('Nenhum data leakage detectado entre TRAIN, VAL e TEST.')


def build_distribution_table(dataset_root: Path) -> tuple[pd.DataFrame, int, int, list[str]]:
    class_names = load_class_names(dataset_root / 'classes.txt')
    image_files = [p for p in sorted((dataset_root / 'images').rglob('*')) if p.suffix.lower() in IMAGE_EXTENSIONS]
    label_files = [p for p in sorted((dataset_root / 'labels').glob('*.txt')) if p.name != 'classes.txt']
    counts = Counter()
    total_objects = 0
    for label_file in label_files:
        class_ids = read_label_classes(label_file)
        total_objects += len(class_ids)
        counts.update(class_ids)
    rows = []
    for class_id, class_name in enumerate(class_names):
        quantity = int(counts.get(class_id, 0))
        percentage = (quantity / total_objects * 100.0) if total_objects else 0.0
        rows.append({'Classe': class_name, 'Quantidade': quantity, 'Percentual': percentage})
    return pd.DataFrame(rows), len(image_files), total_objects, class_names


def plot_distribution(table: pd.DataFrame, split_name: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    axes[0].bar(table['Classe'], table['Quantidade'], color='#005f73')
    axes[0].set_title(f'{split_name} - Distribuicao por classe')
    axes[0].set_ylabel('Quantidade')
    axes[0].tick_params(axis='x', rotation=45)
    axes[1].pie(table['Quantidade'], labels=table['Classe'], autopct='%1.1f%%', startangle=140)
    axes[1].set_title(f'{split_name} - Participacao percentual')
    plt.tight_layout()
    plt.show()


def report_split_distribution(split_root: Path, split_name: str) -> pd.DataFrame:
    table, image_count, annotation_count, _ = build_distribution_table(split_root)
    print(f'[{split_name}] origem: {(PROJECT_ROOT / (split_name.lower() + ".zip")).resolve()}')
    print(f'[{split_name}] imagens: {image_count}')
    print(f'[{split_name}] anotacoes: {annotation_count}')
    display(table)
    plot_distribution(table, split_name)
    return table


TRAIN_ROOT = extract_dataset(TRAIN_ZIP_PATH, TRAIN_DIR)
VAL_ROOT = extract_dataset(VAL_ZIP_PATH, VAL_DIR)
TEST_ROOT = extract_dataset(TEST_ZIP_PATH, TEST_DIR)

print(f'Train extraido apenas de: {TRAIN_ZIP_PATH.resolve()} -> {TRAIN_ROOT.resolve()}')
print(f'Val extraido apenas de: {VAL_ZIP_PATH.resolve()} -> {VAL_ROOT.resolve()}')
print(f'Test extraido apenas de: {TEST_ZIP_PATH.resolve()} -> {TEST_ROOT.resolve()}')

TRAIN_CLASSES = load_class_names(TRAIN_ROOT / 'classes.txt')
VAL_CLASSES = load_class_names(VAL_ROOT / 'classes.txt')
TEST_CLASSES = load_class_names(TEST_ROOT / 'classes.txt')
compare_class_lists(TRAIN_CLASSES, VAL_CLASSES, TEST_CLASSES)
detect_data_leakage(TRAIN_ROOT, VAL_ROOT, TEST_ROOT)

DISTRIBUTION_TRAIN = report_split_distribution(TRAIN_ROOT, 'TRAIN')
DISTRIBUTION_VAL = report_split_distribution(VAL_ROOT, 'VAL')
DISTRIBUTION_TEST = report_split_distribution(TEST_ROOT, 'TEST')

majority_row = DISTRIBUTION_TRAIN.loc[DISTRIBUTION_TRAIN['Quantidade'].idxmax()]
minority_row = DISTRIBUTION_TRAIN.loc[DISTRIBUTION_TRAIN['Quantidade'].idxmin()]
minority_value = float(minority_row['Quantidade'])
majority_value = float(majority_row['Quantidade'])
imbalance_ratio = (majority_value / minority_value) if minority_value else float('inf')

print(f'Total de objetos no TRAIN: {int(DISTRIBUTION_TRAIN["Quantidade"].sum())}')
print(f'Classe majoritaria no TRAIN: {majority_row["Classe"]}')
print(f'Classe minoritaria no TRAIN: {minority_row["Classe"]}')
print(f'Razao de desbalanceamento no TRAIN: {imbalance_ratio:.2f}')

Train extraido apenas de: C:\Users\Enzo\Desktop\rp\train.zip -> C:\Users\Enzo\Desktop\rp\datasets\train
Val extraido apenas de: C:\Users\Enzo\Desktop\rp\val.zip -> C:\Users\Enzo\Desktop\rp\datasets\val
Test extraido apenas de: C:\Users\Enzo\Desktop\rp\test.zip -> C:\Users\Enzo\Desktop\rp\datasets\test
Aviso: possivel data leakage entre TRAIN e VAL por imagem repetida, mas sem conflito de coordenadas iguais.


,tipo,split_a,arquivo_a,split_b,arquivo_b
0,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\8...
1,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\8...
2,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\8...
3,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\e...
4,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\f...
5,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\b...
6,imagem_duplicada,train,C:\Users\Enzo\Desktop\rp\datasets\train\images...,val,C:\Users\Enzo\Desktop\rp\datasets\val\images\b...


[TRAIN] origem: C:\Users\Enzo\Desktop\rp\train.zip
[TRAIN] imagens: 183
[TRAIN] anotacoes: 471


,Classe,Quantidade,Percentual
0,C+,58,12.314225
1,R+,55,11.677282
2,R-,48,10.191083
3,R/CFaltante,57,12.101911
4,Soldagem+,89,18.895966
5,SoldagemCurtoCircuito,48,10.191083
6,SoldagemFaltante,53,11.252654
7,U+,42,8.917197
8,UFaltante,21,4.458599


[VAL] origem: C:\Users\Enzo\Desktop\rp\val.zip
[VAL] imagens: 19
[VAL] anotacoes: 103


...

[TEST] origem: C:\Users\Enzo\Desktop\rp\test.zip
[TEST] imagens: 8
[TEST] anotacoes: 69


C:\Users\Enzo\AppData\Local\Temp\ipykernel_11688\4215971124.py:177: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Classe,Quantidade,Percentual
0,C+,12,17.391304
1,R+,12,17.391304
2,R-,1,1.449275
3,R/CFaltante,13,18.840580
4,Soldagem+,11,15.942029
5,SoldagemCurtoCircuito,1,1.449275
6,SoldagemFaltante,8,11.594203
7,U+,8,11.594203
8,UFaltante,3,4.347826


Total de objetos no TRAIN: 471
Classe majoritaria no TRAIN: Soldagem+
Classe minoritaria no TRAIN: UFaltante
Razao de desbalanceamento no TRAIN: 4.24


C:\Users\Enzo\AppData\Local\Temp\ipykernel_11688\4215971124.py:177: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Célula 3 - Carregamento dos Conjuntos
Recebe train.zip, val.zip e test.zip já separados. Não é realizado split estratificado.

In [3]:
import yaml

CLASS_NAMES = TRAIN_CLASSES
DATASET_YAML = PROJECT_ROOT / 'data.yaml'

dataset_yaml = {
    'path': str(DATASETS_DIR.resolve()),
    'train': str((TRAIN_ROOT / 'images').resolve()),
    'val': str((VAL_ROOT / 'images').resolve()),
    'test': str((TEST_ROOT / 'images').resolve()),
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES,
}


def validate_dataset_yaml(yaml_payload: dict[str, Any], expected_class_names: list[str]) -> None:
    expected_paths = {
        'train': str((TRAIN_ROOT / 'images').resolve()),
        'val': str((VAL_ROOT / 'images').resolve()),
        'test': str((TEST_ROOT / 'images').resolve()),
    }
    for key, expected_value in expected_paths.items():
        if yaml_payload.get(key) != expected_value:
            raise ValueError(f'Dataset YAML inconsistente em {key}: esperado {expected_value}, obtido {yaml_payload.get(key)}')
    if yaml_payload.get('path') != str(DATASETS_DIR.resolve()):
        raise ValueError('Dataset YAML com path incorreto.')
    if int(yaml_payload.get('nc', -1)) != len(expected_class_names):
        raise ValueError('Dataset YAML com nc incorreto.')
    if list(yaml_payload.get('names', [])) != expected_class_names:
        raise ValueError('Dataset YAML com names incorreto.')


validate_dataset_yaml(dataset_yaml, CLASS_NAMES)
with DATASET_YAML.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(dataset_yaml, handle, allow_unicode=True, sort_keys=False)

print('data.yaml criado e validado:')
print(DATASET_YAML.resolve())
print(dataset_yaml)

data.yaml criado e validado:
C:\Users\Enzo\Desktop\rp\data.yaml
{'path': 'C:\\Users\\Enzo\\Desktop\\rp\\datasets', 'train': 'C:\\Users\\Enzo\\Desktop\\rp\\datasets\\train\\images', 'val': 'C:\\Users\\Enzo\\Desktop\\rp\\datasets\\val\\images', 'test': 'C:\\Users\\Enzo\\Desktop\\rp\\datasets\\test\\images', 'nc': 9, 'names': ['C+', 'R+', 'R-', 'R/CFaltante', 'Soldagem+', 'SoldagemCurtoCircuito', 'SoldagemFaltante', 'U+', 'UFaltante']}


## Célula 4 - Treino e Comparação

Aqui estão os controles de modelo, métrica, Optuna e treinamento final.

In [ ]:
# Configurações principais do experimento.
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Mapping

import numpy as np
import pandas as pd
import torchvision

MODEL_NAME = 'YOLO'
USE_OPTUNA = True
N_TRIALS = 10
OPTIMIZATION_METRIC = 'mAP50'

COMPARE_MODELS = ['YOLO', 'T-DETR', 'Faster R-CNN']
ARTIFACTS_DIR = PROJECT_ROOT / 'training_artifacts'
TRAINING_SUMMARY_CSV = PROJECT_ROOT / 'training_summary.csv'
BEST_HYPERPARAMETERS_JSON = PROJECT_ROOT / 'best_hyperparameters.json'
BEST_MODEL_PATH = PROJECT_ROOT / 'best_model.pt'
CONFUSION_MATRIX_DIR = PROJECT_ROOT / 'confusion_matrices'

DEFAULT_HYPERPARAMETERS = {
    'epochs': 60,
    'batch_size': 8,
    'learning_rate': 0.001,
    'weight_decay': 0.0005,
    'optimizer': 'AdamW',
    'img_size': 640,
    'momentum': 0.937,
    'scheduler': 'cosine',
    'patience': 20,
}


@dataclass
class RunArtifacts:
    '''Store the relevant paths and scores for a training run.'''

    model_name: str
    best_metric: float
    best_weights: Path
    best_params: dict[str, Any]


def set_global_seed(seed: int) -> None:
    '''Seed every available backend.'''
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


def resolve_device() -> int:
    '''Require CUDA and use only GPU for training, validation and inference.'''
    if torch is None:
        raise RuntimeError('PyTorch nao esta disponivel. Instale torch com suporte a CUDA antes de treinar.')
    if not torch.cuda.is_available():
        raise RuntimeError('GPU/CUDA obrigatoria: este notebook foi configurado para nao executar em CPU.')
    return 0


def available_workers() -> int:
    '''Keep data-loader workers conservative for notebook environments.'''
    cpu_count = max(1, os.cpu_count() or 1)
    return max(1, min(2, cpu_count // 2))


def metric_lookup(metrics: Any, metric_name: str) -> float:
    '''Normalize metric access across compatible backends.'''
    normalized = metric_name.lower()
    if isinstance(metrics, Mapping) and metric_name in metrics:
        return float(metrics[metric_name])
    if hasattr(metrics, 'results_dict'):
        results = metrics.results_dict
        candidates = {
            'map50': ['metrics/mAP50(B)', 'metrics/mAP50'],
            'map50-95': ['metrics/mAP50-95(B)', 'metrics/mAP50-95'],
            'precision': ['metrics/precision(B)', 'metrics/precision'],
            'recall': ['metrics/recall(B)', 'metrics/recall'],
            'f1': ['metrics/F1(B)', 'metrics/F1'],
        }
        for key in candidates.get(normalized, []):
            if key in results:
                return float(results[key])
    if hasattr(metrics, 'metrics') and isinstance(metrics.metrics, Mapping) and metric_name in metrics.metrics:
        return float(metrics.metrics[metric_name])
    raise KeyError(f'Metrica nao encontrada: {metric_name}')


def sample_optuna_params(trial: Any) -> dict[str, Any]:
    '''Sample the explicit Optuna search space.'''
    return {
        'epochs': trial.suggest_int('epochs', 20, 120, step=10),
        'batch_size': trial.suggest_categorical('batch_size', [4, 8]),
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 0.0, 5e-3),
        'optimizer': trial.suggest_categorical('optimizer', ['SGD', 'Adam', 'AdamW']),
        'img_size': trial.suggest_categorical('img_size', [512, 640]),
        'momentum': trial.suggest_float('momentum', 0.8, 0.99),
        'scheduler': trial.suggest_categorical('scheduler', ['cosine', 'onecycle', 'step']),
        'patience': trial.suggest_int('patience', 10, 30, step=5),
    }


def _ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_labels_for_image(label_file: Path) -> list[list[float]]:
    '''Load YOLO labels as [class, x, y, w, h].'''
    labels: list[list[float]] = []
    if not label_file.exists():
        return labels
    with label_file.open('r', encoding='utf-8') as handle:
        for raw_line in handle:
            raw_line = raw_line.strip()
            if not raw_line:
                continue
            parts = raw_line.split()
            if len(parts) < 5:
                continue
            try:
                labels.append([float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])
            except ValueError:
                continue
    return labels


def yolo_to_xyxy(label_row: list[float], width: int, height: int) -> list[float]:
    class_id, x_center, y_center, box_width, box_height = label_row
    x1 = (x_center - box_width / 2.0) * width
    y1 = (y_center - box_height / 2.0) * height
    x2 = (x_center + box_width / 2.0) * width
    y2 = (y_center + box_height / 2.0) * height
    return [x1, y1, x2, y2, class_id]


def compute_iou(box_a: list[float], box_b: list[float]) -> float:
    '''Compute IoU between two xyxy boxes.'''
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


def collect_ground_truth(split_root: Path) -> dict[str, dict[str, Any]]:
    '''Collect ground-truth boxes and labels for a split.'''
    items: dict[str, dict[str, Any]] = {}
    images_dir = split_root / 'images'
    labels_dir = split_root / 'labels'
    for image_path in sorted(images_dir.rglob('*')):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        with Image.open(image_path) as img:
            width, height = img.size
        label_file = labels_dir / f'{image_path.stem}.txt'
        labels = [yolo_to_xyxy(row, width, height) for row in load_labels_for_image(label_file)]
        items[image_path.stem] = {
            'image_path': image_path,
            'width': width,
            'height': height,
            'labels': labels,
        }
    return items


def _empty_prediction_payload() -> dict[str, list[Any]]:
    return {'boxes': [], 'scores': [], 'labels': []}


def _result_boxes_to_lists(result: Any) -> dict[str, list[Any]]:
    boxes = getattr(result, 'boxes', None)
    if boxes is None:
        return _empty_prediction_payload()
    xyxy = boxes.xyxy.cpu().numpy().tolist() if hasattr(boxes, 'xyxy') else []
    scores = boxes.conf.cpu().numpy().tolist() if hasattr(boxes, 'conf') else []
    labels = boxes.cls.cpu().numpy().tolist() if hasattr(boxes, 'cls') else []
    return {'boxes': xyxy, 'scores': scores, 'labels': labels}


def match_predictions_to_gt(pred_boxes: list[list[float]], pred_labels: list[int], pred_scores: list[float], gt_boxes: list[list[float]], gt_labels: list[int], num_classes: int, iou_threshold: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Greedy matching of predictions to ground truth for one IoU threshold using class-aware matching.'''
    tp = np.zeros(num_classes, dtype=float)
    fp = np.zeros(num_classes, dtype=float)
    fn = np.zeros(num_classes, dtype=float)
    gt_used = [False] * len(gt_boxes)
    order = np.argsort(-np.asarray(pred_scores)) if pred_scores else np.array([], dtype=int)
    for pred_index in order:
        pred_box = pred_boxes[pred_index]
        pred_label = int(pred_labels[pred_index])
        best_iou = 0.0
        best_gt_index = -1
        for gt_index, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
            if gt_used[gt_index] or int(gt_label) != pred_label:
                continue
            current_iou = compute_iou(pred_box, gt_box)
            if current_iou > best_iou:
                best_iou = current_iou
                best_gt_index = gt_index
        if best_gt_index >= 0 and best_iou >= iou_threshold:
            tp[pred_label] += 1
            gt_used[best_gt_index] = True
        else:
            fp[pred_label] += 1
    for gt_index, gt_label in enumerate(gt_labels):
        if not gt_used[gt_index]:
            fn[int(gt_label)] += 1
    return tp, fp, fn


def match_for_confusion(pred_boxes: list[list[float]], pred_labels: list[int], pred_scores: list[float], gt_boxes: list[list[float]], gt_labels: list[int], num_classes: int, iou_threshold: float) -> np.ndarray:
    '''Build a true-vs-predicted confusion matrix using IoU matching.'''
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=float)
    gt_used = [False] * len(gt_boxes)
    order = np.argsort(-np.asarray(pred_scores)) if pred_scores else np.array([], dtype=int)
    for pred_index in order:
        pred_box = pred_boxes[pred_index]
        pred_label = int(pred_labels[pred_index])
        best_iou = 0.0
        best_gt_index = -1
        for gt_index, gt_box in enumerate(gt_boxes):
            if gt_used[gt_index]:
                continue
            current_iou = compute_iou(pred_box, gt_box)
            if current_iou > best_iou:
                best_iou = current_iou
                best_gt_index = gt_index
        if best_gt_index >= 0 and best_iou >= iou_threshold:
            gt_label = int(gt_labels[best_gt_index])
            confusion_matrix[gt_label, pred_label] += 1
            gt_used[best_gt_index] = True
    return confusion_matrix


def evaluate_detections(ground_truth: dict[str, dict[str, Any]], predictions: dict[str, dict[str, list[Any]]], class_names: list[str]) -> tuple[pd.DataFrame, np.ndarray, dict[str, float]]:
    '''Evaluate predictions against ground truth on multiple IoU thresholds.'''
    num_classes = len(class_names)
    iou_thresholds = np.arange(0.5, 0.96, 0.05)
    ap50_per_class = np.zeros(num_classes, dtype=float)
    ap50_95_per_class = np.zeros(num_classes, dtype=float)
    confusion_matrix = np.zeros((num_classes, num_classes), dtype=float)
    base_tp = np.zeros(num_classes, dtype=float)
    base_fp = np.zeros(num_classes, dtype=float)
    base_fn = np.zeros(num_classes, dtype=float)

    for threshold_index, threshold in enumerate(iou_thresholds):
        threshold_tp = np.zeros(num_classes, dtype=float)
        threshold_fp = np.zeros(num_classes, dtype=float)
        threshold_fn = np.zeros(num_classes, dtype=float)
        for stem, gt_data in ground_truth.items():
            pred_data = predictions.get(stem, _empty_prediction_payload())
            pred_boxes = [list(map(float, box)) for box in pred_data['boxes']]
            pred_labels = [int(label) for label in pred_data['labels']]
            pred_scores = [float(score) for score in pred_data['scores']]
            gt_boxes = [box[:4] for box in gt_data['labels']]
            gt_labels = [int(box[4]) for box in gt_data['labels']]
            tp, fp, fn = match_predictions_to_gt(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, num_classes, float(threshold))
            threshold_tp += tp
            threshold_fp += fp
            threshold_fn += fn
            if threshold_index == 0:
                base_tp += tp
                base_fp += fp
                base_fn += fn
                confusion_matrix += match_for_confusion(pred_boxes, pred_labels, pred_scores, gt_boxes, gt_labels, num_classes, threshold)
        for class_id in range(num_classes):
            precision = threshold_tp[class_id] / (threshold_tp[class_id] + threshold_fp[class_id]) if (threshold_tp[class_id] + threshold_fp[class_id]) else 0.0
            recall = threshold_tp[class_id] / (threshold_tp[class_id] + threshold_fn[class_id]) if (threshold_tp[class_id] + threshold_fn[class_id]) else 0.0
            ap_value = precision * recall
            if threshold_index == 0:
                ap50_per_class[class_id] = ap_value
            ap50_95_per_class[class_id] += ap_value

    ap50_95_per_class /= float(len(iou_thresholds))
    precision = float(base_tp.sum() / (base_tp.sum() + base_fp.sum())) if (base_tp.sum() + base_fp.sum()) else 0.0
    recall = float(base_tp.sum() / (base_tp.sum() + base_fn.sum())) if (base_tp.sum() + base_fn.sum()) else 0.0
    f1 = float((2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0)
    metrics = {
        'Accuracy': float(np.trace(confusion_matrix) / confusion_matrix.sum()) if confusion_matrix.sum() else 0.0,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'mAP50': float(np.mean(ap50_per_class)) if ap50_per_class.size else 0.0,
        'mAP50-95': float(np.mean(ap50_95_per_class)) if ap50_95_per_class.size else 0.0,
    }
    rows = []
    for class_id, class_name in enumerate(class_names):
        rows.append({'Classe': class_name, 'AP50': float(ap50_per_class[class_id]), 'AP50-95': float(ap50_95_per_class[class_id])})
    return pd.DataFrame(rows), confusion_matrix, metrics

def save_confusion_outputs(cm: np.ndarray, class_names: list[str], model_name: str) -> None:
    '''Save confusion matrix as CSV and PNG.'''
    _ensure_dir(CONFUSION_MATRIX_DIR)
    slug = model_name.replace(' ', '_').lower()
    csv_path = CONFUSION_MATRIX_DIR / f'{slug}_confusion_matrix.csv'
    png_path = CONFUSION_MATRIX_DIR / f'{slug}_confusion_matrix.png'
    frame = pd.DataFrame(cm, index=class_names, columns=class_names)
    frame.to_csv(csv_path, index=True, encoding='utf-8')
    plt.figure(figsize=(12, 10))
    plt.imshow(cm, cmap='Blues')
    plt.title(f'Matriz de confusao - {model_name}')
    plt.colorbar()
    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha='right')
    plt.yticks(ticks, class_names)
    plt.xlabel('Predito')
    plt.ylabel('Real')
    for row_index in range(cm.shape[0]):
        for col_index in range(cm.shape[1]):
            plt.text(col_index, row_index, f'{cm[row_index, col_index]:.0f}', ha='center', va='center', color='black')
    plt.tight_layout()
    plt.savefig(png_path, dpi=200, bbox_inches='tight')
    plt.show()


class YoloBackend:
    '''Backend wrapper for Ultralytics YOLO.'''

    def __init__(self, model_name: str, class_names: list[str], dataset_yaml: Path, project_dir: Path) -> None:
        from ultralytics import YOLO

        self.model_name = model_name
        self.YOLO = YOLO
        self.class_names = class_names
        self.dataset_yaml = dataset_yaml
        self.project_dir = project_dir

    def train(self, params: Mapping[str, Any], run_name: str) -> Path:
        model = self.YOLO(self._resolve_weights())
        result = model.train(
            data=str(self.dataset_yaml),
            epochs=int(params['epochs']),
            batch=int(params['batch_size']),
            imgsz=int(params['img_size']),
            lr0=float(params['learning_rate']),
            weight_decay=float(params['weight_decay']),
            optimizer=str(params['optimizer']),
            momentum=float(params['momentum']),
            patience=int(params['patience']),
            device=resolve_device(),
            workers=available_workers(),
            project=str(self.project_dir),
            name=run_name,
            exist_ok=True,
            cache=False,
            amp=True,
            verbose=False,
            val=True,
            save=True,
        )
        return Path(result.save_dir) / 'weights' / 'best.pt'

    def evaluate(self, weights: Path, split: str = 'val', imgsz: int = 640) -> Any:
        model = self.YOLO(str(weights))
        return model.val(data=str(self.dataset_yaml), split=split, imgsz=imgsz, device=resolve_device(), verbose=False)

    def predict(self, weights: Path, source: Path, save_dir: Path) -> dict[str, dict[str, list[Any]]]:
        model = self.YOLO(str(weights))
        results = model.predict(source=str(source), save=False, project=str(save_dir), exist_ok=True, device=resolve_device(), verbose=False)
        predictions: dict[str, dict[str, list[Any]]] = {}
        for result in results:
            stem = Path(result.path).stem
            predictions[stem] = _result_boxes_to_lists(result)
        return predictions

    def _resolve_weights(self) -> str:
        return 'yolov8s.pt'


class TDETRBackend:
    '''Backend wrapper for Ultralytics RT-DETR style models.'''

    def __init__(self, model_name: str, class_names: list[str], dataset_yaml: Path, project_dir: Path) -> None:
        try:
            from ultralytics import RTDETR  # type: ignore
            self.Model = RTDETR
        except Exception:
            from ultralytics import YOLO  # type: ignore
            self.Model = YOLO
        self.model_name = model_name
        self.class_names = class_names
        self.dataset_yaml = dataset_yaml
        self.project_dir = project_dir

    def train(self, params: Mapping[str, Any], run_name: str) -> Path:
        model = self.Model(self._resolve_weights())
        result = model.train(
            data=str(self.dataset_yaml),
            epochs=int(params['epochs']),
            batch=int(params['batch_size']),
            imgsz=int(params['img_size']),
            lr0=float(params['learning_rate']),
            momentum=float(params['momentum']),
            patience=int(params['patience']),
            device=resolve_device(),
            workers=available_workers(),
            project=str(self.project_dir),
            name=run_name,
            exist_ok=True,
            cache=False,
            amp=True,
            verbose=False,
            val=True,
            save=True,
        )
        return Path(result.save_dir) / 'weights' / 'best.pt'

    def evaluate(self, weights: Path, split: str = 'val', imgsz: int = 640) -> Any:
        model = self.Model(str(weights))
        return model.val(data=str(self.dataset_yaml), split=split, imgsz=imgsz, device=resolve_device(), verbose=False)

    def predict(self, weights: Path, source: Path, save_dir: Path) -> dict[str, dict[str, list[Any]]]:
        model = self.Model(str(weights))
        results = model.predict(source=str(source), save=False, project=str(save_dir), exist_ok=True, device=resolve_device(), verbose=False)
        predictions: dict[str, dict[str, list[Any]]] = {}
        for result in results:
            stem = Path(result.path).stem
            predictions[stem] = _result_boxes_to_lists(result)
        return predictions

    def _resolve_weights(self) -> str:
        return 'rtdetr-l.pt'


class FasterRCNNDataset(torch.utils.data.Dataset):
    '''Torchvision dataset for YOLO labels.'''

    def __init__(self, split_root: Path, class_names: list[str]) -> None:
        self.split_root = split_root
        self.class_names = class_names
        self.image_paths = [path for path in sorted((split_root / 'images').rglob('*')) if path.suffix.lower() in IMAGE_EXTENSIONS]

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, index: int) -> tuple[Any, dict[str, torch.Tensor]]:
        image_path = self.image_paths[index]
        label_path = self.split_root / 'labels' / f'{image_path.stem}.txt'
        image = Image.open(image_path).convert('RGB')
        width, height = image.size
        raw_labels = load_labels_for_image(label_path)
        boxes = []
        labels = []
        for row in raw_labels:
            xyxy = yolo_to_xyxy(row, width, height)
            boxes.append(xyxy[:4])
            labels.append(int(xyxy[4]) + 1)
        image_tensor = torchvision.transforms.functional.to_tensor(image)
        target = {
            'boxes': torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
            'image_id': torch.tensor([index]),
            'area': torch.tensor([(box[2] - box[0]) * (box[3] - box[1]) for box in boxes], dtype=torch.float32) if boxes else torch.zeros((0,), dtype=torch.float32),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),
        }
        return image_tensor, target


def faster_rcnn_model(num_classes: int) -> torch.nn.Module:
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)
    return model


class FasterRCNNBackend:
    '''Real Faster R-CNN backend with training, validation, inference and evaluation.'''

    def __init__(self, model_name: str, class_names: list[str], dataset_yaml: Path, project_dir: Path) -> None:
        self.model_name = model_name
        self.class_names = class_names
        self.dataset_yaml = dataset_yaml
        self.project_dir = project_dir
        self.device = torch.device(f'cuda:{resolve_device()}')

    def train(self, params: Mapping[str, Any], run_name: str) -> Path:
        train_dataset = FasterRCNNDataset(TRAIN_ROOT, self.class_names)
        train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=int(params['batch_size']), shuffle=True, num_workers=available_workers(), collate_fn=lambda batch: tuple(zip(*batch)))
        model = faster_rcnn_model(len(self.class_names) + 1).to(self.device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=float(params['learning_rate']), weight_decay=float(params['weight_decay']))
        lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
        best_metric = -1.0
        run_dir = _ensure_dir(self.project_dir / run_name)
        best_path = run_dir / 'weights' / 'best.pt'
        _ensure_dir(best_path.parent)
        patience = int(params['patience'])
        epochs_without_improvement = 0

        for _epoch in range(int(params['epochs'])):
            model.train()
            for images, targets in train_loader:
                images = [image.to(self.device) for image in images]
                targets = [{key: value.to(self.device) for key, value in target.items()} for target in targets]
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                optimizer.zero_grad()
                losses.backward()
                optimizer.step()
            lr_scheduler.step()
            val_metrics = self.evaluate_model(model, VAL_ROOT)
            current_metric = val_metrics['mAP50']
            if current_metric > best_metric:
                best_metric = current_metric
                torch.save(model.state_dict(), best_path)
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
        return best_path

    def load_model(self, weights: Path) -> torch.nn.Module:
        model = faster_rcnn_model(len(self.class_names) + 1).to(self.device)
        state_dict = torch.load(weights, map_location=self.device)
        model.load_state_dict(state_dict)
        model.eval()
        return model

    def predict(self, weights: Path, source: Path, save_dir: Path) -> dict[str, dict[str, list[Any]]]:
        model = self.load_model(weights)
        _ensure_dir(save_dir)
        predictions: dict[str, dict[str, list[Any]]] = {}
        for image_path in sorted(source.rglob('*')):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            image = Image.open(image_path).convert('RGB')
            tensor = torchvision.transforms.functional.to_tensor(image).to(self.device)
            with torch.no_grad():
                output = model([tensor])[0]
            predictions[image_path.stem] = {
                'boxes': output['boxes'].detach().cpu().numpy().tolist(),
                'scores': output['scores'].detach().cpu().numpy().tolist(),
                'labels': [int(label) - 1 for label in output['labels'].detach().cpu().numpy().tolist()],
            }
        return predictions

    def evaluate_model(self, model: torch.nn.Module, split_root: Path) -> dict[str, float]:
        model.eval()
        ground_truth = collect_ground_truth(split_root)
        predictions: dict[str, dict[str, list[Any]]] = {}
        for image_path in sorted((split_root / 'images').rglob('*')):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            image = Image.open(image_path).convert('RGB')
            tensor = torchvision.transforms.functional.to_tensor(image).to(self.device)
            with torch.no_grad():
                output = model([tensor])[0]
            predictions[image_path.stem] = {
                'boxes': output['boxes'].detach().cpu().numpy().tolist(),
                'scores': output['scores'].detach().cpu().numpy().tolist(),
                'labels': [int(label) - 1 for label in output['labels'].detach().cpu().numpy().tolist()],
            }
        _, cm, metrics = evaluate_detections(ground_truth, predictions, self.class_names)
        metrics['confusion_matrix'] = cm
        return metrics

    def evaluate(self, weights: Path, split: str = 'val', imgsz: int = 640) -> Any:
        if split != 'val':
            raise ValueError('Faster R-CNN usa evaluate() apenas em VAL; TEST fica restrito a evaluate_model_on_test().')
        model = self.load_model(weights)
        split_root = VAL_ROOT
        ground_truth = collect_ground_truth(split_root)
        predictions = self.predict(weights, split_root / 'images', self.project_dir / 'predictions' / split)
        _, cm, metrics = evaluate_detections(ground_truth, predictions, self.class_names)
        metrics['confusion_matrix'] = cm
        return metrics


def build_backend(model_name: str, class_names: list[str], dataset_yaml: Path, project_dir: Path) -> Any:
    '''Create the backend for the selected model.'''
    if model_name == 'YOLO':
        return YoloBackend(model_name=model_name, class_names=class_names, dataset_yaml=dataset_yaml, project_dir=project_dir)
    if model_name == 'T-DETR':
        return TDETRBackend(model_name=model_name, class_names=class_names, dataset_yaml=dataset_yaml, project_dir=project_dir)
    if model_name == 'Faster R-CNN':
        return FasterRCNNBackend(model_name=model_name, class_names=class_names, dataset_yaml=dataset_yaml, project_dir=project_dir)
    raise NotImplementedError(f'Nenhum backend foi registrado para {model_name}. Registre o adaptador em build_backend().')


def run_hyperparameter_search(model_name: str, backend_factory: Callable[[Path], Any], metric_name: str) -> tuple[dict[str, Any], float, Path]:
    '''Search hyperparameters using TRAIN for fitting and VAL for model selection.'''
    try:
        import optuna
    except Exception as exc:
        raise ImportError('A biblioteca optuna nao esta disponivel.') from exc

    study = optuna.create_study(direction='maximize')

    def objective(trial: Any) -> float:
        params = sample_optuna_params(trial)
        backend = backend_factory(DATASET_YAML)
        weights = backend.train(params=params, run_name=f'{model_name}_trial_{trial.number}')
        val_metrics = backend.evaluate(weights=weights, split='val', imgsz=int(params['img_size']))
        trial.set_user_attr('weights', str(weights))
        return metric_lookup(val_metrics, metric_name)

    study.optimize(objective, n_trials=N_TRIALS)
    best_params = DEFAULT_HYPERPARAMETERS.copy()
    best_params.update(study.best_trial.params)
    best_weights = Path(study.best_trial.user_attrs['weights'])
    return best_params, float(study.best_value), best_weights


def train_final_model(model_name: str, backend_factory: Callable[[Path], Any], params: Mapping[str, Any]) -> RunArtifacts:
    '''Train the selected model with TRAIN and select using VAL.'''
    backend = backend_factory(DATASET_YAML)
    weights = backend.train(params=params, run_name=f'{model_name}_final')
    val_metrics = backend.evaluate(weights=weights, split='val', imgsz=int(params['img_size']))
    best_metric = metric_lookup(val_metrics, OPTIMIZATION_METRIC)
    return RunArtifacts(model_name=model_name, best_metric=best_metric, best_weights=weights, best_params=dict(params))


def save_json(payload: Mapping[str, Any], path: Path) -> None:
    '''Persist a JSON payload.'''
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, default=str)


def save_summary(rows: list[dict[str, Any]], path: Path) -> None:
    '''Persist the training summary table.'''
    pd.DataFrame(rows).to_csv(path, index=False)


def run_model_workflow(model_name: str) -> RunArtifacts:
    '''Run hyperparameter selection and final training for a single model.'''
    backend_factory = lambda dataset_yaml: build_backend(model_name, CLASS_NAMES, dataset_yaml, ARTIFACTS_DIR / model_name)
    best_params, best_score, best_weights = run_hyperparameter_search(model_name, backend_factory, OPTIMIZATION_METRIC)
    print(f'Melhor score em VAL para {model_name}: {best_score:.4f}')
    print(f'Melhores hiperparametros em {model_name}: {best_params}')
    final_artifact = train_final_model(model_name, backend_factory, best_params)
    save_json({'model_name': model_name, 'best_metric': final_artifact.best_metric, 'best_params': final_artifact.best_params, 'best_weights': str(final_artifact.best_weights)}, BEST_HYPERPARAMETERS_JSON.with_name(f"best_hyperparameters_{model_name.replace(' ', '_').lower()}.json"))
    return final_artifact


set_global_seed(42)
validate_dataset_yaml(dataset_yaml, CLASS_NAMES)
resolve_device()

artifacts: list[RunArtifacts] = []
summary_rows: list[dict[str, Any]] = []
for candidate_model in COMPARE_MODELS:
    result = run_model_workflow(candidate_model)
    artifacts.append(result)
    summary_rows.append({
        'Modelo': result.model_name,
        'Melhor_Metrica_VAL': result.best_metric,
        'Hiperparametros': json.dumps(result.best_params, ensure_ascii=False),
        'Peso': str(result.best_weights),
    })

ARTIFACTS_BY_MODEL = {artifact.model_name: artifact for artifact in artifacts}

best_artifact = max(artifacts, key=lambda item: item.best_metric)
_ensure_dir(ARTIFACTS_DIR)
shutil.copy2(best_artifact.best_weights, BEST_MODEL_PATH)
save_json({'model_name': best_artifact.model_name, 'best_metric': best_artifact.best_metric, 'best_params': best_artifact.best_params, 'best_weights': str(best_artifact.best_weights)}, BEST_HYPERPARAMETERS_JSON)
save_summary(summary_rows, TRAINING_SUMMARY_CSV)

print('Melhor Trial')
print(best_artifact.model_name)
print('Melhor Metrica de validacao')
print(best_artifact.best_metric)
print('Melhores Hiperparametros')
print(best_artifact.best_params)
print(f'Peso salvo em: {BEST_MODEL_PATH.resolve()}')
print(f'Summary salvo em: {TRAINING_SUMMARY_CSV.resolve()}')

c:\Users\Enzo\Desktop\rp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-12 00:20:38,666] A new study created in memory with name: no-name-44f8a736-a18c-40e7-b9b4-f7a21ad89f01


Ultralytics 8.4.66  Python-3.11.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Enzo\Desktop\rp\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0020360812711581913, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.9707665793723292, mosaic=1.0, multi_scale=0.0, name=YOLO_trial_0, nbs=64, nms=False, opset=None, optimize=Fal

[I 2026-06-12 00:40:36,166] Trial 0 finished with value: 0.38637304351991897 and parameters: {'epochs': 100, 'batch_size': 8, 'learning_rate': 0.0020360812711581913, 'weight_decay': 0.0029314107754144394, 'optimizer': 'Adam', 'img_size': 640, 'momentum': 0.9707665793723292, 'scheduler': 'step', 'patience': 15}. Best is trial 0 with value: 0.38637304351991897.


Ultralytics 8.4.66  Python-3.11.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1050 Ti, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Enzo\Desktop\rp\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0022984631761375825, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.8109160923867552, mosaic=1.0, multi_scale=0.0, name=YOLO_trial_1, nbs=64, nms=False, opset=None, optimize=Fals

[I 2026-06-12 00:55:04,556] Trial 1 finished with value: 0.3840744846412368 and parameters: {'epochs': 80, 'batch_size': 8, 'learning_rate': 0.0022984631761375825, 'weight_decay': 0.0012386046016817083, 'optimizer': 'AdamW', 'img_size': 512, 'momentum': 0.8109160923867552, 'scheduler': 'onecycle', 'patience': 30}. Best is trial 0 with value: 0.38637304351991897.


...

     14/120      5.92G     0.5458        1.1    0.05958          1       1280: 100% ━━━━━━━━━━━━ 92/92 20.7s/it 31:4816.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.1it/s 2.7s1.9ss
                   all         19        103      0.689      0.208      0.137     0.0705

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


c:\Users\Enzo\Desktop\rp\.venv\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     15/120         6G     0.5578      1.018    0.06055         19       1280: 100% ━━━━━━━━━━━━ 92/92 20.4s/it 31:1313.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.3it/s 2.2s1.7s
                   all         19        103      0.112      0.283      0.175        0.1

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


c:\Users\Enzo\Desktop\rp\.venv\Lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     16/120      5.93G     0.5355     0.9727     0.0554         46       1280: 60% ━━━━━━━───── 56/92 30.1s/it 26:37<18:04

## Célula 5 - Avaliação Final

Aqui são calculadas as métricas finais no conjunto de teste e, se possível, geradas predições qualitativas.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def load_best_artifact(summary_path: Path) -> tuple[str, Path, dict[str, Any]]:
    if not summary_path.exists():
        raise FileNotFoundError(f'Training summary nao encontrado: {summary_path.resolve()}')
    table = pd.read_csv(summary_path)
    if table.empty:
        raise ValueError('training_summary.csv esta vazio.')
    best_row = table.sort_values('Melhor_Metrica_VAL', ascending=False).iloc[0]
    return str(best_row['Modelo']), Path(str(best_row['Peso'])), json.loads(str(best_row['Hiperparametros']))


def metrics_to_row(model_name: str, metrics: dict[str, float]) -> dict[str, float | str]:
    return {
        'Modelo': model_name,
        'Accuracy': metrics.get('Accuracy', 0.0),
        'Precision': metrics.get('Precision', 0.0),
        'Recall': metrics.get('Recall', 0.0),
        'F1-Score': metrics.get('F1-Score', 0.0),
        'mAP50': metrics.get('mAP50', 0.0),
        'mAP50-95': metrics.get('mAP50-95', 0.0),
    }


def evaluate_model_on_test(model_name: str, backend: Any, weights: Path, class_names: list[str]) -> tuple[dict[str, float | str], pd.DataFrame, np.ndarray, dict[str, float]]:
    ground_truth = collect_ground_truth(TEST_ROOT)
    predictions = backend.predict(weights, TEST_ROOT / 'images', PROJECT_ROOT / 'test_predictions' / model_name.replace(' ', '_').lower())
    ap_table, confusion_matrix, metrics = evaluate_detections(ground_truth, predictions, class_names)
    save_confusion_outputs(confusion_matrix, class_names, model_name)
    ap_table = ap_table.copy()
    ap_table.insert(0, 'Modelo', model_name)
    return metrics_to_row(model_name, metrics), ap_table, confusion_matrix, metrics


best_model_name, best_weights, best_params = load_best_artifact(TRAINING_SUMMARY_CSV)
print(f'Melhor modelo encontrado por validacao: {best_model_name}')
print(f'Melhores hiperparametros: {best_params}')
print(f'Caminho do peso salvo: {best_weights.resolve()}')

metric_rows: list[dict[str, float | str]] = []
per_class_tables: list[pd.DataFrame] = []
evaluation_details: dict[str, dict[str, Any]] = {}

for candidate_model in COMPARE_MODELS:
    artifact = ARTIFACTS_BY_MODEL[candidate_model]
    backend = build_backend(candidate_model, CLASS_NAMES, DATASET_YAML, ARTIFACTS_DIR / candidate_model)
    metric_row, ap_table, confusion_matrix, metrics = evaluate_model_on_test(candidate_model, backend, artifact.best_weights, CLASS_NAMES)
    metric_rows.append(metric_row)
    per_class_tables.append(ap_table)
    evaluation_details[candidate_model] = {
        'metrics': metrics,
        'per_class_ap': ap_table,
        'confusion_matrix': confusion_matrix,
    }

final_metrics_table = pd.DataFrame(metric_rows).sort_values('mAP50', ascending=False).reset_index(drop=True)
per_class_metrics_table = pd.concat(per_class_tables, ignore_index=True)

FINAL_METRICS_CSV = PROJECT_ROOT / 'final_test_metrics_by_model.csv'
FINAL_PER_CLASS_CSV = PROJECT_ROOT / 'final_test_per_class_metrics.csv'
final_metrics_table.to_csv(FINAL_METRICS_CSV, index=False, encoding='utf-8')
per_class_metrics_table.to_csv(FINAL_PER_CLASS_CSV, index=False, encoding='utf-8')

display(final_metrics_table)
display(per_class_metrics_table)

print('Resumo final')
print(f'Melhor modelo por VAL: {best_model_name}')
print(f'Melhores hiperparametros por VAL: {best_params}')
print(f'Peso salvo: {best_weights.resolve()}')
print(f'Metricas finais por modelo: {FINAL_METRICS_CSV.resolve()}')
print(f'Metricas finais por classe: {FINAL_PER_CLASS_CSV.resolve()}')
print(f'Matrizes de confusao: {CONFUSION_MATRIX_DIR.resolve()}')
print('TEST foi usado exclusivamente na avaliacao final.')
print('TEST nao participa do treinamento, da busca Optuna nem da selecao de hiperparametros.')